In [1]:
%pip -q install lightgbm xgboost catboost category-encoders joblib


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.2/99.2 MB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.9/85.9 kB 6.6 MB/s eta 0:00:00


In [2]:
# نصب کتابخانه‌های لازم (در صورت نیاز)
!pip install xgboost lightgbm scikit-learn pandas numpy

In [3]:


import pandas as pd
import numpy as np
import warnings
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
import xgboost as xgb
import lightgbm as lgb

# نادیده گرفتن هشدارهای غیرضروری
warnings.filterwarnings('ignore')

# 1. بارگذاری داده‌ها
df = pd.read_csv('IKCO_Saipa_Cars_final.csv')

# 2. پاکسازی اولیه
# حذف ستون‌های توضیحات متنی که مدل نمی‌تواند پردازش کند (چون فیچرهای باینری آن‌ها را داریم)
cols_to_drop = ['description', 'body_status', 'engine_status', 'chassis_status']
df_clean = df.drop(columns=[c for c in cols_to_drop if c in df.columns])

# تبدیل داده‌های متنی (Categorical) به عدد
label_encoders = {}
# Add 'year' to categorical_cols to handle non-numeric year entries
categorical_cols = ['title', 'color', 'brand_type', 'gearbox', 'fuel_type', 'price_class', 'year']

for col in categorical_cols:
    if col in df_clean.columns:
        le = LabelEncoder()
        df_clean[col] = le.fit_transform(df_clean[col].astype(str))
        label_encoders[col] = le

# NEW STEP: Impute missing values in numerical columns before defining X and y
for col in df_clean.columns:
    if df_clean[col].isnull().any():
        if pd.api.types.is_numeric_dtype(df_clean[col]):
            df_clean[col] = df_clean[col].fillna(df_clean[col].median())
        else: # Handle any remaining non-numeric columns with NaNs by filling with a placeholder string
            df_clean[col] = df_clean[col].fillna('missing')

# 3. تعیین X (ویژگی‌ها) و y (قیمت)
X = df_clean.drop('price', axis=1)
y = df_clean['price']

# 4. تقسیم داده‌ها به آموزش و تست
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 5. استانداردسازی داده‌ها (برای مدل‌هایی مثل SVM و KNN و Linear حیاتی است)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("✅ Data Preprocessing Completed.")
print(f"Features used: {list(X.columns)}")

✅ Data Preprocessing Completed.
Features used: ['title', 'mileage_km', 'year', 'color', 'brand_type', 'gearbox', 'fuel_type', 'price_class', 'درب رنگ', 'درب تعویض', 'درب بدون رنگ', 'کاپوت تعویض', 'کاپوت بدون رنگ', 'درب صندوق رنگ', 'درب صندوق بدون رنگ', 'شاسی ضرب\u200cخوردگی', 'شاسی پلمپ', 'شاسی سالم', 'سقف رنگ', 'سقف بدون رنگ', 'ستون سالم', 'ستون ضرب\u200cخوردگی', 'گلگیر رنگ', 'گلگیر بدون رنگ', 'گلگیر تعویض', 'فنی سالم', 'فنی نیاز به تعمیر', 'فنی تعمیر شده', 'گیربکس سالم', 'گیربکس نیاز به تعمیر', 'یک لکه رنگ', 'دو لکه رنگ', 'دور رنگ']


In [4]:
# تعریف مدل‌ها و فضای جستجوی هایپرپارامترها
models_params = {
    'Linear Regression': {
        'model': LinearRegression(),
        'params': {} # رگرسیون خطی ساده پارامتر خاصی ندارد
    },
    'Ridge': {
        'model': Ridge(),
        'params': {'alpha': [0.1, 1.0, 10.0, 100.0]}
    },
    'Lasso': {
        'model': Lasso(),
        'params': {'alpha': [0.1, 1.0, 10.0, 100.0]}
    },
    'Decision Tree': {
        'model': DecisionTreeRegressor(),
        'params': {'max_depth': [None, 10, 20, 30], 'min_samples_split': [2, 5, 10]}
    },
    'Random Forest': {
        'model': RandomForestRegressor(),
        'params': {'n_estimators': [50, 100, 200], 'max_depth': [None, 10, 20], 'min_samples_split': [2, 5]}
    },
    'XGBoost': {
        'model': xgb.XGBRegressor(objective='reg:squarederror'),
        'params': {'n_estimators': [100, 200], 'learning_rate': [0.01, 0.1, 0.2], 'max_depth': [3, 5, 7]}
    },
    'LightGBM': {
        'model': lgb.LGBMRegressor(verbose=-1),
        'params': {'n_estimators': [100, 200], 'learning_rate': [0.01, 0.1], 'num_leaves': [31, 50]}
    },
    'SVR': { # ماشین بردار پشتیبان
        'model': SVR(),
        'params': {'C': [0.1, 1, 10], 'kernel': ['linear', 'rbf']}
    },
    'KNN': { # نزدیکترین همسایه
        'model': KNeighborsRegressor(),
        'params': {'n_neighbors': [3, 5, 7, 9], 'weights': ['uniform', 'distance']}
    }
}

trained_models = {}
best_params_found = {}

print("🚀 Starting Hyperparameter Tuning (This may take a while)...")

for name, mp in models_params.items():
    print(f"training {name}...")
    # از داده‌های اسکیل شده استفاده می‌کنیم
    clf = RandomizedSearchCV(mp['model'], mp['params'], cv=3, n_iter=10, scoring='neg_mean_squared_error', n_jobs=-1, random_state=42)
    clf.fit(X_train_scaled, y_train)

    trained_models[name] = clf.best_estimator_
    best_params_found[name] = clf.best_params_

print("\n✅ All models trained and tuned successfully!")

🚀 Starting Hyperparameter Tuning (This may take a while)...
training Linear Regression...
training Ridge...
training Lasso...
training Decision Tree...
training Random Forest...
training XGBoost...
training LightGBM...
training SVR...
training KNN...

✅ All models trained and tuned successfully!


In [5]:
from sklearn.metrics import mean_squared_error, mean_absolute_error

error_metrics = []

for name, model in trained_models.items():
    y_pred = model.predict(X_test_scaled)

    mse = mean_squared_error(y_test, y_pred)
    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mse)

    error_metrics.append({
        'Model': name,
        'MAE (Mean Absolute Error)': mae,
        'MSE (Mean Squared Error)': mse,
        'RMSE (Root MSE)': rmse
    })

df_errors = pd.DataFrame(error_metrics).sort_values(by='RMSE (Root MSE)')
print("📊 جدول مقایسه خطاها (هرچه کمتر، بهتر):")
display(df_errors)

📊 جدول مقایسه خطاها (هرچه کمتر، بهتر):


,Model,MAE (Mean Absolute Error),MSE (Mean Squared Error),RMSE (Root MSE)
8,KNN,6.953251e+07,5.760576e+16,2.400120e+08
4,Random Forest,5.620137e+07,6.116148e+16,2.473085e+08
6,LightGBM,7.414785e+07,6.229938e+16,2.495984e+08
5,XGBoost,1.193258e+08,7.114729e+16,2.667345e+08
1,Ridge,1.045061e+08,7.317961e+16,2.705173e+08
2,Lasso,1.063729e+08,7.330995e+16,2.707581e+08
0,Linear Regression,1.063730e+08,7.330997e+16,2.707581e+08
3,Decision Tree,6.117685e+07,1.041994e+17,3.227993e+08
7,SVR,2.506284e+08,1.559391e+17,3.948912e+08


In [6]:
from sklearn.metrics import r2_score

accuracy_metrics = []

for name, model in trained_models.items():
    y_pred = model.predict(X_test_scaled)

    r2 = r2_score(y_test, y_pred)
    # محاسبه MAPE (میانگین درصد خطای مطلق)
    mape = np.mean(np.abs((y_test - y_pred) / y_test)) * 100

    accuracy_metrics.append({
        'Model': name,
        'R2 Score (Darat)': r2, # نزدیک به 1 یعنی عالی
        'MAPE (%)': mape        # میانگین درصد خطا
    })

df_accuracy = pd.DataFrame(accuracy_metrics).sort_values(by='R2 Score (Darat)', ascending=False)
print("📊 جدول مقایسه دقت (R2 هرچه به 1 نزدیکتر بهتر):")
display(df_accuracy)

# انتخاب بهترین مدل برای استفاده در مرحله بعد
best_model_name = df_accuracy.iloc[0]['Model']
best_model = trained_models[best_model_name]
print(f"\n🏆 بهترین مدل انتخاب شده: {best_model_name}")

📊 جدول مقایسه دقت (R2 هرچه به 1 نزدیکتر بهتر):


,Model,R2 Score (Darat),MAPE (%)
8,KNN,0.627095,15.785413
4,Random Forest,0.604078,10.287622
6,LightGBM,0.596712,16.836100
5,XGBoost,0.539436,32.634082
1,Ridge,0.526280,21.189225
2,Lasso,0.525436,21.541269
0,Linear Regression,0.525436,21.541310
3,Decision Tree,0.325477,12.074188
7,SVR,-0.009454,65.047445



🏆 بهترین مدل انتخاب شده: KNN


In [7]:
# انتخاب یک نمونه تصادفی از داده‌های تست
random_idx = np.random.choice(len(X_test))
sample_input = X_test.iloc[[random_idx]]
sample_input_scaled = X_test_scaled[[random_idx]] # استفاده از نسخه اسکیل شده
actual_price = y_test.iloc[random_idx]

# پیش‌بینی با بهترین مدل
predicted_price = best_model.predict(sample_input_scaled)[0]

print(f"🔍 **Sample Prediction Test**")
print("-" * 30)
print(f"🚗 Car Details (Encoded): \n{sample_input.to_dict(orient='records')[0]}")
print("-" * 30)
print(f"💰 قیمت واقعی: {actual_price:,.0f} تومان")
print(f"🤖 قیمت پیش‌بینی شده: {predicted_price:,.0f} تومان")
print(f"📉 اختلاف: {abs(actual_price - predicted_price):,.0f} تومان")

🔍 **Sample Prediction Test**
------------------------------
🚗 Car Details (Encoded): 
{'title': 2196, 'mileage_km': 40000, 'year': 33, 'color': 17, 'brand_type': 88, 'gearbox': 26, 'fuel_type': 7, 'price_class': 2, 'درب رنگ': 0.0, 'درب تعویض': 0.0, 'درب بدون رنگ': 0.0, 'کاپوت تعویض': 0.0, 'کاپوت بدون رنگ': 0.0, 'درب صندوق رنگ': 0.0, 'درب صندوق بدون رنگ': 0.0, 'شاسی ضرب\u200cخوردگی': 0.0, 'شاسی پلمپ': 1.0, 'شاسی سالم': 0.0, 'سقف رنگ': 0.0, 'سقف بدون رنگ': 0.0, 'ستون سالم': 0.0, 'ستون ضرب\u200cخوردگی': 0.0, 'گلگیر رنگ': 0.0, 'گلگیر بدون رنگ': 0.0, 'گلگیر تعویض': 0.0, 'فنی سالم': 0.0, 'فنی نیاز به تعمیر': 0.0, 'فنی تعمیر شده': 0.0, 'گیربکس سالم': 0.0, 'گیربکس نیاز به تعمیر': 0.0, 'یک لکه رنگ': 0.0, 'دو لکه رنگ': 0.0, 'دور رنگ': 0.0}
------------------------------
💰 قیمت واقعی: 970,000,000 تومان
🤖 قیمت پیش‌بینی شده: 975,785,306 تومان
📉 اختلاف: 5,785,306 تومان


In [10]:
def predict_car_price(year, mileage, model_name, gearbox_type,
                      is_chassis_sealed=1, is_technical_healthy=1,
                      has_painted_door=0, has_painted_hood=0, has_painted_roof=0):
    """
    تابع پیش‌بینی قیمت بر اساس صدمات و مشخصات
    ورودی‌های 0 و 1 برای وضعیت سلامت: 1 یعنی سالم/پلمپ، 0 یعنی ضربه/رنگ
    """
    # 1. ساخت یک دیکشنری با تمام ستون‌های X
    input_data = {col: 0 for col in X.columns}

    # 2. پر کردن مقادیر عددی ساده
    input_data['year'] = year
    input_data['mileage_km'] = mileage

    # 3. پر کردن مقادیر انکود شده (Categorical)
    # نکته: در یک اپلیکیشن واقعی باید try/except بگذارید برای مقادیر ناشناخته
    try:
        input_data['title'] = label_encoders['title'].transform([model_name])[0]
        input_data['gearbox'] = label_encoders['gearbox'].transform([gearbox_type])[0]
        # مقادیر پیش‌فرض برای سایر کتگوری‌ها (می‌توانید دقیق‌تر کنید)
        input_data['brand_type'] = label_encoders['brand_type'].transform([label_encoders['brand_type'].classes_[0]])[0]
        input_data['fuel_type'] = label_encoders['fuel_type'].transform(['بنزینی'])[0]
        input_data['color'] = label_encoders['color'].transform([label_encoders['color'].classes_[0]])[0]
        input_data['price_class'] = 0 # فرض
    except Exception as e:
        print(f"Error in encoding input: {e}")
        return

    # 4. پر کردن صدمات (بر اساس ستون‌های موجود در دیتاست شما)
    # نام ستون‌ها باید دقیقاً مثل دیتاست باشد
    if 'شاسی پلمپ' in input_data: input_data['شاسی پلمپ'] = is_chassis_sealed
    if 'فنی سالم' in input_data: input_data['فنی سالم'] = is_technical_healthy
    if 'درب رنگ' in input_data: input_data['درب رنگ'] = has_painted_door
    if 'کاپوت رنگ' in input_data: input_data['کاپوت رنگ'] = has_painted_hood # اگر نام ستون متفاوت است تغییر دهید
    if 'سقف رنگ' in input_data: input_data['سقف رنگ'] = has_painted_roof

    # 5. تبدیل به DataFrame
    input_df = pd.DataFrame([input_data])

    # 6. اسکیل کردن ورودی
    input_scaled = scaler.transform(input_df)

    # 7. پیش‌بینی
    price = best_model.predict(input_scaled)[0]
    return price

# --- فرم ورودی تست ---
print("🚗 **Car Price Estimator**")
print("لطفاً مشخصات زیر را برای تست تغییر دهید:")

estimated_price = predict_car_price(
    year=1399,              # سال ساخت
    mileage=50000,          # کارکرد
    model_name="ساینا دنده‌ای EX", # باید دقیقاً یکی از نام‌های موجود در دیتاست باشد
    gearbox_type="دنده‌ای",    # نوع گیربکس

    # --- وضعیت صدمات (1=بله/سالم، 0=خیر/ضربه) ---
    is_chassis_sealed=1,    # آیا شاسی پلمپ است؟ (1: بله)
    is_technical_healthy=1, # آیا فنی سالم است؟ (1: بله)
    has_painted_door=0,     # آیا درب رنگ دارد؟ (1: بله، 0: خیر)
    has_painted_hood=0,     # آیا کاپوت رنگ دارد؟
    has_painted_roof=0      # آیا سقف رنگ دارد؟
)

if estimated_price:
    print(f"\n💎 قیمت تخمینی خودرو: {estimated_price:,.0f} تومان")

🚗 **Car Price Estimator**
لطفاً مشخصات زیر را برای تست تغییر دهید:
Error in encoding input: y contains previously unseen labels: 'ساینا دنده\u200cای EX'
